# Task 1 — Data Quality Pipeline

**Architecture:** Class-based `DataQualityPipeline` with Plotly visualisations,
KNN imputation, IP validation via the `ipaddress` stdlib module,
and automated EDA via ydata-profiling.

| Component | Technology |
|-----------|------------|
| Visualisation | Plotly (interactive) |
| Imputation | KNNImputer (sklearn) |
| IP Validation | `ipaddress` stdlib |
| Automated EDA | ydata-profiling HTML |

**Datasets:** `data/task1.csv` (network traffic) · `data/features.csv` (column descriptions)

In [7]:
# Install required libraries
!pip install plotly ydata-profiling scikit-learn pandas numpy --quiet


[notice] A new release of pip is available: 25.3 -> 26.1
[notice] To update, run: pip install --upgrade pip


In [ ]:
# ── Standard library ──────────────────────────────────────────────────────────
import os
import ipaddress
import warnings

# ── Third-party ───────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.impute import KNNImputer
from IPython.display import display, IFrame

warnings.filterwarnings('ignore')

# ── Data path resolution ──────────────────────────────────────────────────────
# __vsc_ipynb_file__ is set by the VS Code kernel to the notebook's absolute path.
# Falls back to a sibling data/ folder relative to CWD when not available.
try:
    _NB_DIR = os.path.dirname(os.path.abspath(__vsc_ipynb_file__))
except NameError:
    _NB_DIR = os.getcwd()

def find_data_dir():
    '''Locate the data/ directory relative to the notebook, with CWD fallbacks.'''
    candidates = [
        os.path.join(_NB_DIR, '..', 'data'),          # sibling data/ folder
        os.path.join(os.getcwd(), 'data'),             # project root fallback
        os.path.join(_NB_DIR, 'data'),
        'data', '../data', '../../data',               # last-resort fallbacks
    ]
    for path in candidates:
        full = os.path.normpath(os.path.abspath(path))
        if os.path.isfile(os.path.join(full, 'task1.csv')):
            return full
    raise FileNotFoundError('Cannot locate the data/ directory containing task1.csv')

DATA_DIR = find_data_dir()
print(f'Data directory resolved: {DATA_DIR}')

Data directory resolved: /workspaces/2026-4-6-Data-Science-for-Cyber-Security/V2/data


In [ ]:
class DataQualityPipeline:
    '''
    Class-based data quality pipeline for network traffic analysis.

    Attributes:
        csv_path (str)       : Path to the raw network traffic CSV.
        features_path (str)  : Path to the feature description CSV.
        df (pd.DataFrame)    : DataFrame built up through pipeline steps.

    Methods:
        load()                 -- Load CSV files into memory.
        validate_missing()     -- Detect and impute missing values via KNN.
        validate_ips()         -- Validate IP columns via ipaddress module.
        validate_timestamps()  -- Parse/validate Stime and Ltime columns.
        analyse_distribution() -- Produce Plotly distribution charts.
        run_all()              -- Execute the full pipeline end-to-end.
    '''

    def __init__(self, csv_path: str, features_path: str):
        self.csv_path = csv_path
        self.features_path = features_path
        self.df = None
        self.features_df = None

    # ── Step 1: Load ─────────────────────────────────────────────────────────
    def load(self) -> pd.DataFrame:
        '''Load the raw CSV and optional feature descriptor file.'''
        self.df = pd.read_csv(self.csv_path, low_memory=False)
        try:
            self.features_df = pd.read_csv(self.features_path)
            print(f'  Feature descriptions loaded: {len(self.features_df)} entries')
        except Exception as e:
            print(f'  Warning – could not load features file: {e}')
        print(f'  Rows: {len(self.df):,}  |  Columns: {self.df.shape[1]}')
        display(self.df.head(3))
        return self.df

    # ── Step 2: Missing values ────────────────────────────────────────────────
    def validate_missing(self) -> pd.DataFrame:
        '''
        Identify missing values and apply KNN imputation to numeric columns.
        Categorical NaNs are filled with the placeholder string "Unknown".
        '''
        mc = self.df.isnull().sum()
        mp = (mc / len(self.df) * 100).round(2)
        summary = (
            pd.DataFrame({'Missing Count': mc, 'Missing %': mp})
            .query('`Missing Count` > 0')
            .sort_values('Missing %', ascending=False)
        )
        print(f'  Columns with missing values: {len(summary)}')
        display(summary)

        if not summary.empty:
            fig = px.bar(
                summary.reset_index().rename(columns={'index': 'Column'}),
                x='Column', y='Missing %',
                title='Missing Value Percentage by Column',
                labels={'Column': 'Column Name', 'Missing %': 'Missing (%)'},
                color='Missing %',
                color_continuous_scale='Reds',
                template='plotly_dark'
            )
            fig.update_layout(xaxis_tickangle=-45, height=420)
            fig.show()

        # KNN imputation for all numeric columns
        numeric_cols = self.df.select_dtypes(include=[np.number]).columns.tolist()
        if numeric_cols:
            imputer = KNNImputer(n_neighbors=5)
            self.df[numeric_cols] = imputer.fit_transform(self.df[numeric_cols])
            print(f'  KNN imputation applied to {len(numeric_cols)} numeric columns (k=5).')

        # Fill remaining categorical NaNs
        for col in self.df.select_dtypes(include='object').columns:
            if self.df[col].isnull().any():
                self.df[col].fillna('Unknown', inplace=True)

        print('  Missing value handling complete.')
        return self.df

    # ── Step 3: IP validation ─────────────────────────────────────────────────
    def validate_ips(self) -> pd.DataFrame:
        '''
        Validate source (srcip) and destination (dstip) IP address columns
        using Python stdlib ipaddress module – no regular expressions used.
        Each address is classified as correct, missing, or error.
        '''
        def classify_ip(val):
            '''Return status label for a single IP value.'''
            if pd.isna(val) or str(val).strip() == '':
                return 'missing'
            try:
                ipaddress.ip_address(str(val).strip())
                return 'correct'
            except ValueError:
                return 'error'

        records = []
        for col, label in [('srcip', 'Source IP'), ('dstip', 'Destination IP')]:
            if col not in self.df.columns:
                continue
            statuses = self.df[col].apply(classify_ip)
            self.df[f'{col}_status'] = statuses
            for status, cnt in statuses.value_counts().items():
                records.append({'IP Column': label, 'Status': status, 'Count': int(cnt)})
            print(f'  {col}: {statuses.value_counts().to_dict()}')

        if records:
            summary_df = pd.DataFrame(records)
            fig = px.bar(
                summary_df, x='IP Column', y='Count', color='Status',
                barmode='group',
                title='IP Address Validation — Source vs Destination IP',
                color_discrete_map={
                    'correct': '#2ecc71',
                    'missing': '#f39c12',
                    'error': '#e74c3c'
                },
                template='plotly_dark',
                text='Count'
            )
            fig.update_traces(textposition='outside')
            fig.update_layout(
                xaxis_title='IP Column',
                yaxis_title='Number of Records',
                legend_title='Status',
                height=450
            )
            fig.show()

        return self.df

    # ── Step 4: Timestamp validation ──────────────────────────────────────────
    def validate_timestamps(self) -> pd.DataFrame:
        '''
        Parse Stime and Ltime Unix epoch columns to datetime.
        Reports invalid timestamps and visualises hourly packet distribution.
        '''
        for ts_col in ['Stime', 'Ltime']:
            if ts_col not in self.df.columns:
                continue
            self.df[ts_col] = pd.to_numeric(self.df[ts_col], errors='coerce')
            self.df[f'{ts_col}_dt'] = pd.to_datetime(
                self.df[ts_col], unit='s', errors='coerce'
            )
            n_invalid = int(self.df[f'{ts_col}_dt'].isna().sum())
            print(f'  {ts_col}: {n_invalid} invalid/null timestamps')

        if 'Stime_dt' in self.df.columns:
            temp = self.df.dropna(subset=['Stime_dt']).copy()
            temp['Hour'] = temp['Stime_dt'].dt.hour
            hour_df = (
                temp['Hour'].value_counts()
                .sort_index()
                .reset_index()
                .rename(columns={'index': 'Hour', 'Hour': 'Packet Count'})
            )
            hour_df.columns = ['Hour', 'Packet Count']
            fig = px.bar(
                hour_df, x='Hour', y='Packet Count',
                title='Packet Count by Hour of Day (Stime)',
                labels={'Hour': 'Hour (0-23)', 'Packet Count': 'Number of Packets'},
                color='Packet Count',
                color_continuous_scale='Blues',
                template='plotly_dark'
            )
            fig.update_layout(height=400)
            fig.show()

        return self.df

    # ── Step 5: Distribution analysis ─────────────────────────────────────────
    def analyse_distribution(self) -> None:
        '''
        Produce two Plotly-based distribution analyses:
          1. Protocol distribution — Donut pie chart
          2. Attack category distribution — Horizontal bar chart
        '''
        # Analysis 1: Protocol distribution
        if 'proto' in self.df.columns:
            proto_df = (
                self.df['proto']
                .value_counts()
                .head(10)
                .reset_index()
            )
            proto_df.columns = ['Protocol', 'Count']
            fig1 = px.pie(
                proto_df, names='Protocol', values='Count',
                title='Top 10 Network Protocol Distribution',
                template='plotly_dark',
                hole=0.4
            )
            fig1.update_traces(textposition='inside', textinfo='percent+label')
            fig1.update_layout(legend_title='Protocol')
            fig1.show()

        # Analysis 2: Attack category / traffic label
        cat_col = 'attack_cat' if 'attack_cat' in self.df.columns else 'Label'
        if cat_col in self.df.columns:
            cat_df = self.df[cat_col].value_counts().reset_index()
            cat_df.columns = ['Category', 'Count']
            cat_df['Type'] = cat_df['Category'].apply(
                lambda x: 'Benign'
                if str(x).strip().lower() in ['', 'benign', '0', 'normal']
                else 'Attack'
            )
            fig2 = px.bar(
                cat_df, y='Category', x='Count',
                orientation='h',
                title='Traffic Distribution by Attack Category',
                color='Type',
                color_discrete_map={'Benign': '#2ecc71', 'Attack': '#e74c3c'},
                template='plotly_dark',
                text='Count'
            )
            fig2.update_traces(textposition='outside')
            fig2.update_layout(
                xaxis_title='Number of Records',
                yaxis_title='Category',
                yaxis={'categoryorder': 'total ascending'},
                height=500,
                legend_title='Traffic Type'
            )
            fig2.show()

    # ── Run all steps ─────────────────────────────────────────────────────────
    def run_all(self) -> pd.DataFrame:
        '''Execute all pipeline steps in sequence.'''
        print('=' * 58)
        print('  DATA QUALITY PIPELINE  |  Class-Based Architecture')
        print('=' * 58)
        print('\n[Step 1] Loading data...')
        self.load()
        print('\n[Step 2] Validating missing values...')
        self.validate_missing()
        print('\n[Step 3] Validating IP addresses...')
        self.validate_ips()
        print('\n[Step 4] Validating timestamps...')
        self.validate_timestamps()
        print('\n[Step 5] Analysing distributions...')
        self.analyse_distribution()
        print('\n' + '=' * 58)
        print(f'  DONE — final shape: {self.df.shape[0]:,} rows x {self.df.shape[1]} cols')
        return self.df

In [10]:
# ── Instantiate and run the full pipeline ────────────────────────────────────
pipeline = DataQualityPipeline(
    csv_path=os.path.join(DATA_DIR, 'task1.csv'),
    features_path=os.path.join(DATA_DIR, 'features.csv')
)

cleaned_df = pipeline.run_all()

  DATA QUALITY PIPELINE v2  |  Class-Based Architecture

[Step 1] Loading data...
  Warning – could not load features file: 'utf-8' codec can't decode byte 0x92 in position 1620: invalid start byte
  Rows: 9,967  |  Columns: 49


,srcip,sport,dstip,dsport,proto,state,dur,sbytes,dbytes,sttl,...,ct_ftp_cmd,ct_srv_src,ct_srv_dst,ct_dst_ltm,ct_src_ ltm,ct_src_dport_ltm,ct_dst_sport_ltm,ct_dst_src_ltm,attack_cat,Label
0,175.45.176.1,1043.0,149.171.126.14,53,udp,INT,0.000009,114,0,254,...,,13,13,13,14,13,13,13,Generic,1
1,59.166.0.5,40332.0,149.171.126.5,11493,tcp,FIN,0.344959,4160,2976,31,...,,4,7,4,2,1,1,2,NaN,0
2,175.45.176.0,47439.0,149.171.126.10,53,udp,INT,0.000009,114,0,254,...,,24,24,24,24,24,13,24,Generic,1



[Step 2] Validating missing values...
  Columns with missing values: 5


,Missing Count,Missing %
is_ftp_login,9807,98.39
ct_flw_http_mthd,9275,93.06
attack_cat,7994,80.20
srcip,57,0.57
sport,51,0.51


  KNN imputation applied to 41 numeric columns (k=5).
  Missing value handling complete.

[Step 3] Validating IP addresses...
  srcip: {'correct': 9891, 'error': 76}
  dstip: {'correct': 9966, 'error': 1}



[Step 4] Validating timestamps...
  Stime: 0 invalid/null timestamps
  Ltime: 0 invalid/null timestamps



[Step 5] Analysing distributions...



  DONE — final shape: 9,967 rows x 53 cols


## Automated EDA Report
ydata-profiling generates a single-page interactive HTML report with
univariate statistics, correlations, and data quality alerts.

In [11]:
# ── Automated EDA Report via ydata-profiling ─────────────────────────────────
try:
    from ydata_profiling import ProfileReport

    print('Generating automated EDA report (this may take ~30 s)...')
    profile = ProfileReport(
        pipeline.df.head(3000),        # sample for performance
        title='Network Traffic — Automated EDA Report',
        explorative=True,
        minimal=True                   # faster generation
    )
    report_path = os.path.join(os.getcwd(), 'eda_report.html')
    profile.to_file(report_path)
    print(f'EDA report saved: {report_path}')

    # Display inline
    display(IFrame(src='eda_report.html', width='100%', height=650))

except ImportError:
    print('ydata-profiling not available. Install with: pip install ydata-profiling')
except Exception as exc:
    print(f'EDA report generation skipped: {exc}')

Generating automated EDA report (this may take ~30 s)...


Summarize dataset:  45%|████▍     | 26/58 [00:00<00:00, 128.73it/s, Describe variable: Djit]      

Export report to file: 100%|██████████| 1/1 [00:00<00:00, 146.91it/s]

EDA report saved: /workspaces/2026-4-6-Data-Science-for-Cyber-Security/V2/Task1/eda_report.html


## Summary

| Step | Technique | Output |
|------|-----------|--------|
| Load | `pd.read_csv` | Raw DataFrame |
| Missing values | KNNImputer (k=5) | Imputed DataFrame |
| IP validation | `ipaddress.ip_address()` | Status column per IP field |
| Timestamps | `pd.to_datetime(unit='s')` | Hourly distribution chart |
| Distribution | Plotly pie + bar charts | Protocol & attack category |
| EDA | ydata-profiling | `eda_report.html` |